# QBE: sync the rules to the rules table

Writes the rules in the YAML files of one config to the rules table set under `rules.table` (`dq_rules`), one
row per rule, as the rows of the config's `rules.rule_set`. The `config` parameter picks it:

| `config` | YAML files | `rule_set` | Read by |
|---|---|---|---|
| `validation.yml` (default) | `config/tables` | `validation` | `qbe_validation.ipynb` |
| `clean_validation.yml` | `config/clean_tables` | `clean_validation` | `clean_validation.ipynb` |

Run it after changing the YAML files; each validation notebook reads the table when its `rules.source` is `table`.

The rule set's rows are **replaced** on each run, so they always match the YAML files: rules removed from the YAML
are removed from the table, and edits made directly in the table are overwritten. The other rule sets' rows are
left as they are. Rows synced before the table had a `rule_set` column are `validation`'s. `${name}` placeholders (e.g.
`${source_catalog}`) are stored as written and resolved by `qbe_validation.ipynb` when it runs.

**Parameters:** `config`, and the `variables` in that config that the rules table's name uses (e.g. `results_catalog`).
Needs only PyYAML (no DQX).

In [ ]:
import re
import sys
from pathlib import Path

import yaml
from pyspark.sql import functions as F

# Config paths are relative to use_cases/qbe, the parent of this notebook's folder (notebooks); its src folder
# holds the com.qbe.de package
SRC_DIR = Path.cwd().parent
sys.path.insert(0, str(SRC_DIR / "src"))
from com.qbe.de.rules import DEFAULT_RULE_SET, RULES_SCHEMA, read_yaml_tables, to_rule_rows

# The config whose rules are synced (a file in config): validation.yml or clean_validation.yml
try:
    dbutils.widgets.text("config", "validation.yml")
    config_file = dbutils.widgets.get("config")
except NameError:                   # no dbutils outside Databricks: use the default
    config_file = "validation.yml"
raw_settings = yaml.safe_load((SRC_DIR / "config" / config_file).read_text())["validation"]
rules_table = raw_settings["rules"]["table"]
rule_set = raw_settings["rules"].get("rule_set", DEFAULT_RULE_SET)
tables_dir = raw_settings["rules"].get("tables_dir", "config/tables")
if not re.fullmatch(r"[a-z_]+", rule_set):
    raise ValueError(f"rules.rule_set in {config_file} must be lowercase letters and _, got: {rule_set}")
# Widgets only for the variables in the rules table's name (e.g. results_catalog)
variables = {}
for name, default in (raw_settings.get("variables") or {}).items():
    if f"${{{name}}}" not in rules_table:
        continue
    try:
        dbutils.widgets.text(name, str(default))
        variables[name] = dbutils.widgets.get(name)
    except NameError:               # no dbutils outside Databricks: use the default
        variables[name] = str(default)

for name, value in variables.items():
    rules_table = rules_table.replace(f"${{{name}}}", value)

rows = to_rule_rows(read_yaml_tables(SRC_DIR / tables_dir), rule_set)
rules = spark.createDataFrame(rows, RULES_SCHEMA).withColumn("synced_ts", F.current_timestamp())
if spark.catalog.tableExists(rules_table):
    if "rule_set" not in spark.read.table(rules_table).columns:
        # Synced before rule_set was a column: those rows are qbe_validation's
        spark.sql(f"ALTER TABLE {rules_table} ADD COLUMNS (rule_set STRING)")
        spark.sql(f"UPDATE {rules_table} SET rule_set = '{DEFAULT_RULE_SET}' WHERE rule_set IS NULL")
    # Only this rule set's rows are replaced; mergeSchema adds the columns newer rows have
    (rules.write.mode("overwrite").option("replaceWhere", f"rule_set = '{rule_set}'").option("mergeSchema", "true")
     .saveAsTable(rules_table))
else:
    rules.write.saveAsTable(rules_table)

print(f"{len(rows)} rules from {tables_dir} synced to {rules_table} as rule_set {rule_set}")
display(spark.read.table(rules_table).where(F.col("rule_set") == rule_set)
        .select("rule_set", "table_name", "column_name", "rule_id", "check_function", "enabled"))